# 🧹 Data Cleaning trong Pandas — Hướng dẫn từ Cơ bản đến Nâng cao

Tài liệu này tổng hợp và **hệ thống hóa lại** toàn bộ nội dung về chủ đề **Data Cleaning (làm sạch dữ liệu)** dựa trên các bài viết tham khảo từ GeeksforGeeks:

1. [Handling Missing Data](https://www.geeksforgeeks.org/data-analysis/working-with-missing-data-in-pandas/) — Xử lý dữ liệu thiếu
2. [Removing Duplicates](https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-drop_duplicates/) — Loại bỏ dữ liệu trùng lặp
3. [Pandas Change Datatype](https://www.geeksforgeeks.org/pandas/pandas-change-datatype/) — Thay đổi kiểu dữ liệu
4. [Drop Empty Columns in Pandas](https://www.geeksforgeeks.org/python/drop-empty-columns-in-pandas/) — Xóa cột rỗng
5. [String manipulations in Pandas](https://www.geeksforgeeks.org/python/string-manipulations-in-pandas-dataframe/) — Thao tác chuỗi
6. [String methods in Pandas](https://www.geeksforgeeks.org/python/top-10-string-methods-in-pandas/) — Top phương thức chuỗi
7. [Detect Mixed Data Types and Fix it](https://www.geeksforgeeks.org/python/pandas-detect-mixed-data-types-and-fix-it/) — Phát hiện & sửa kiểu dữ liệu hỗn hợp

> Mỗi phần đều có: **lý thuyết → ví dụ minh họa → bài tập thực hành** trên bộ dữ liệu thật `employees.csv` (1000 nhân viên, có sẵn missing values, cột rỗng một phần, kiểu dữ liệu cần chuẩn hóa...).

## 📚 Mục lục

0. [Chuẩn bị dữ liệu](#0)
1. [Xử lý dữ liệu thiếu (Missing Data)](#1)
2. [Loại bỏ dữ liệu trùng lặp (Duplicates)](#2)
3. [Thay đổi kiểu dữ liệu (Change Datatype)](#3)
4. [Xóa cột rỗng (Drop Empty Columns)](#4)
5. [Thao tác chuỗi (String Manipulations)](#5)
6. [Top các phương thức chuỗi thường dùng](#6)
7. [Phát hiện & xử lý kiểu dữ liệu hỗn hợp (Mixed Data Types)](#7)
8. [🏆 Bài tập tổng hợp (Capstone Project)](#8)
9. [Tổng kết & Checklist làm sạch dữ liệu](#9)


<a id="0"></a>
## 0. Chuẩn bị dữ liệu

Chúng ta sẽ sử dụng file `employees.csv` — dữ liệu về 1000 nhân viên với các cột:

| Cột | Ý nghĩa |
|---|---|
| `First Name` | Tên nhân viên |
| `Gender` | Giới tính |
| `Start Date` | Ngày bắt đầu làm việc |
| `Last Login Time` | Giờ đăng nhập gần nhất |
| `Salary` | Lương |
| `Bonus %` | Phần trăm thưởng |
| `Senior Management` | Có phải quản lý cấp cao không |
| `Team` | Phòng ban |

Đây là bộ dữ liệu **"bẩn"** điển hình: có giá trị thiếu, có cột thiếu nhiều dữ liệu, kiểu dữ liệu ngày/giờ đang ở dạng chuỗi... rất phù hợp để luyện tập.


In [ ]:
import pandas as pd
import numpy as np

# Đọc dữ liệu
df = pd.read_csv("employees.csv")

print("Kích thước dữ liệu:", df.shape)
df.head()


In [ ]:
# Xem tổng quan thông tin của DataFrame: kiểu dữ liệu, số lượng non-null...
df.info()


In [ ]:
# Thống kê mô tả nhanh cho các cột số
df.describe()


<a id="1"></a>
## 1. Xử lý dữ liệu thiếu (Handling Missing Data)

Dữ liệu thiếu (missing data) trong Pandas thường được biểu diễn bằng `NaN` (Not a Number), `None`, hoặc `NaT` (Not a Time) đối với cột kiểu ngày giờ.

### Nguyên nhân dữ liệu bị thiếu
- Lỗi khi nhập liệu / thu thập dữ liệu
- Dữ liệu bị mất trong quá trình truyền tải
- Người khảo sát không trả lời một số câu hỏi
- Ghép nối (merge/join) dữ liệu từ nhiều nguồn không khớp nhau

### 1.1 Phát hiện dữ liệu thiếu

Các hàm chính:
- `df.isna()` / `df.isnull()` → trả về DataFrame kiểu bool, `True` nếu là giá trị thiếu
- `df.notna()` / `df.notnull()` → ngược lại với `isna()`
- Kết hợp với `.sum()` để đếm số lượng giá trị thiếu theo từng cột


In [ ]:
# Đếm số giá trị thiếu theo từng cột
df.isna().sum()


In [ ]:
# Tỷ lệ (%) dữ liệu thiếu theo từng cột — rất hữu ích khi quyết định xóa hay giữ cột
(df.isna().sum() / len(df) * 100).round(2).sort_values(ascending=False)


In [ ]:
# Kiểm tra xem 1 dòng có giá trị thiếu nào không
df[df.isna().any(axis=1)].head()


### 1.2 Loại bỏ dữ liệu thiếu — `dropna()`

Cú pháp: `df.dropna(axis=0, how='any', thresh=None, subset=None, inplace=False)`

| Tham số | Ý nghĩa |
|---|---|
| `axis` | `0` = xóa theo hàng, `1` = xóa theo cột |
| `how` | `'any'` = xóa nếu có ít nhất 1 NaN, `'all'` = chỉ xóa nếu **toàn bộ** là NaN |
| `thresh` | Giữ lại hàng/cột có ít nhất `thresh` giá trị không phải NaN |
| `subset` | Chỉ xét NaN trên các cột được liệt kê |


In [ ]:
# Xóa toàn bộ các hàng có ít nhất 1 giá trị thiếu (cách "mạnh tay" nhất)
df_dropped_any = df.dropna(how='any')
print("Số dòng còn lại:", len(df_dropped_any), "/", len(df))


In [ ]:
# Chỉ xóa hàng nếu thiếu ở cột quan trọng, ví dụ 'First Name'
df_dropna_subset = df.dropna(subset=['First Name'])
print("Số dòng còn lại:", len(df_dropna_subset))


In [ ]:
# Giữ lại các hàng có ít nhất 6 giá trị không NaN (trên tổng 8 cột)
df_thresh = df.dropna(thresh=6)
print("Số dòng còn lại:", len(df_thresh))


### 1.3 Điền giá trị thiếu — `fillna()`

Thay vì xóa (làm mất dữ liệu), ta thường **điền** giá trị thiếu bằng một giá trị hợp lý.

Cú pháp: `df.fillna(value=None, method=None, axis=None, limit=None)`

Các chiến lược điền phổ biến:
- Điền bằng **giá trị cố định** (ví dụ: `"Unknown"`, `0`)
- Điền bằng **giá trị thống kê**: `mean()`, `median()`, `mode()`
- Điền theo hướng: `ffill` (forward fill — lấy giá trị phía trước), `bfill` (backward fill — lấy giá trị phía sau)


In [ ]:
# Điền giá trị thiếu cho cột phân loại (categorical) bằng 1 giá trị cố định
df_fill = df.copy()
df_fill['Team'] = df_fill['Team'].fillna('Unknown')
df_fill['Gender'] = df_fill['Gender'].fillna('Not Specified')

df_fill[['Team', 'Gender']].isna().sum()


In [ ]:
# Điền giá trị thiếu cho cột số bằng trung bình (mean) hoặc trung vị (median)
df_fill2 = df.copy()
mean_salary = df_fill2['Salary'].mean()
df_fill2['Salary'] = df_fill2['Salary'].fillna(mean_salary)
print("Lương trung bình dùng để điền:", round(mean_salary, 2))


In [ ]:
# fillna với method='ffill' (lấy giá trị liền trước) và 'bfill' (lấy giá trị liền sau)
demo = pd.Series([1, np.nan, np.nan, 4, np.nan, 6])
print("Gốc:   ", demo.tolist())
print("ffill: ", demo.ffill().tolist())
print("bfill: ", demo.bfill().tolist())


### 1.4 Nội suy giá trị thiếu — `interpolate()`

Dùng cho dữ liệu **có tính liên tục / xu hướng** (ví dụ: chuỗi thời gian, dữ liệu tăng dần), Pandas sẽ ước lượng giá trị thiếu dựa trên các giá trị lân cận.


In [ ]:
demo2 = pd.Series([10, np.nan, np.nan, 40, 50])
print("Gốc:          ", demo2.tolist())
print("Nội suy tuyến tính:", demo2.interpolate(method='linear').tolist())


### ✏️ 1.5 Bài tập thực hành — Missing Data

Sử dụng DataFrame `df` (bộ dữ liệu `employees.csv`) để thực hiện:

1. In ra danh sách các cột có giá trị thiếu, sắp xếp theo số lượng thiếu giảm dần.
2. Điền cột `Gender` bằng giá trị xuất hiện nhiều nhất (mode).
3. Điền cột `Senior Management` bằng `False` (giả định nếu thiếu thông tin thì không phải quản lý cấp cao).
4. Điền cột `Bonus %` bằng trung vị (median) của toàn bộ cột.
5. Sau khi xử lý xong 4 bước trên, kiểm tra lại `isna().sum()` — cột nào vẫn còn thiếu? Vì sao?

> 💡 Gợi ý: dùng `.mode()[0]` để lấy giá trị xuất hiện nhiều nhất.


In [ ]:
# TODO: Bài tập 1.5 — viết code của bạn tại đây
df_ex1 = df.copy()


# 1. Cột có giá trị thiếu, sắp xếp giảm dần


# 2. Điền Gender bằng mode


# 3. Điền Senior Management bằng False


# 4. Điền Bonus % bằng median


# 5. Kiểm tra lại


<a id="2"></a>
## 2. Loại bỏ dữ liệu trùng lặp (Removing Duplicates)

Dữ liệu trùng lặp thường xảy ra khi thu thập dữ liệu từ nhiều nguồn, khi merge dữ liệu, hoặc do lỗi nhập liệu 2 lần.

### 2.1 Phát hiện dòng trùng lặp — `duplicated()`

Cú pháp: `df.duplicated(subset=None, keep='first')`

- `subset`: chỉ xét trùng lặp trên các cột được chỉ định (mặc định là toàn bộ cột)
- `keep`:
  - `'first'` (mặc định): đánh dấu các bản sao *sau* bản đầu tiên là trùng lặp
  - `'last'`: đánh dấu các bản sao *trước* bản cuối cùng là trùng lặp
  - `False`: đánh dấu **tất cả** các bản ghi trùng lặp (kể cả bản đầu tiên)


In [ ]:
# Đếm tổng số dòng trùng lặp hoàn toàn (toàn bộ các cột giống nhau)
print("Số dòng trùng lặp:", df.duplicated().sum())


In [ ]:
# Tạo dữ liệu minh họa có trùng lặp để dễ quan sát
demo_df = pd.DataFrame({
    'Name': ['An', 'Binh', 'An', 'Chi', 'Binh'],
    'Team': ['Sales', 'IT', 'Sales', 'HR', 'IT']
})
demo_df


In [ ]:
print("keep='first' (mặc định):")
print(demo_df.duplicated().tolist())

print("\nkeep='last':")
print(demo_df.duplicated(keep='last').tolist())

print("\nkeep=False (đánh dấu tất cả bản trùng):")
print(demo_df.duplicated(keep=False).tolist())


### 2.2 Xóa dòng trùng lặp — `drop_duplicates()`

Cú pháp: `df.drop_duplicates(subset=None, keep='first', inplace=False)` — tham số giống hệt `duplicated()`.


In [ ]:
# Xóa các dòng trùng lặp hoàn toàn, chỉ giữ lại bản đầu tiên
demo_clean = demo_df.drop_duplicates()
demo_clean


In [ ]:
# Xóa trùng lặp chỉ dựa trên cột 'Team' (giữ 1 người đại diện mỗi Team)
demo_df.drop_duplicates(subset=['Team'], keep='first')


### ✏️ 2.3 Bài tập thực hành — Duplicates

1. Kiểm tra xem `employees.csv` có dòng nào trùng lặp hoàn toàn không.
2. Giả sử ta muốn xem: mỗi `Team` có bao nhiêu người trùng **cùng First Name** — hãy tìm các dòng có `First Name` trùng nhau (dùng `subset=['First Name']`, `keep=False`) và đếm số lượng.
3. Tạo một bản DataFrame mới `df_unique_team`, trong đó với mỗi `Team`, chỉ giữ lại nhân viên có `Salary` cao nhất (gợi ý: sort theo Salary giảm dần trước, sau đó `drop_duplicates(subset=['Team'], keep='first')`).


In [ ]:
# TODO: Bài tập 2.3 — viết code của bạn tại đây

# 1. Kiểm tra trùng lặp hoàn toàn


# 2. Các dòng trùng First Name


# 3. Nhân viên lương cao nhất mỗi Team


<a id="3"></a>
## 3. Thay đổi kiểu dữ liệu (Pandas Change Datatype)

Kiểu dữ liệu đúng giúp việc tính toán, lọc, và phân tích chính xác hơn (ví dụ: nếu ngày tháng vẫn là `object` (chuỗi) thì không thể tính toán được khoảng thời gian).

### 3.1 Kiểm tra kiểu dữ liệu hiện tại


In [ ]:
df.dtypes


### 3.2 `astype()` — Chuyển đổi kiểu dữ liệu tường minh

Cú pháp: `df['col'] = df['col'].astype('kiểu_mới')`

Kiểu phổ biến: `int`, `float`, `str`, `bool`, `category`, `'datetime64[ns]'`


In [ ]:
df_type = df.copy()

# Chuyển 'Senior Management' về kiểu bool thực sự (tránh lỗi khi có NaN, dùng thêm .astype('boolean') hỗ trợ NA)
df_type['Senior Management'] = df_type['Senior Management'].astype('boolean')

# Chuyển 'Team' về kiểu category giúp tiết kiệm bộ nhớ với dữ liệu phân loại lặp lại nhiều
df_type['Team'] = df_type['Team'].astype('category')

df_type.dtypes


### 3.3 `pd.to_numeric()` — Chuyển sang kiểu số

Rất hữu ích khi cột số bị lưu dưới dạng chuỗi (`object`), có thể lẫn ký tự không hợp lệ.

Tham số quan trọng: `errors='raise' | 'coerce' | 'ignore'`
- `'coerce'`: giá trị không chuyển được sẽ thành `NaN` thay vì báo lỗi


In [ ]:
messy_numbers = pd.Series(['100', '200', 'abc', '300.5', None])
print("Gốc:", messy_numbers.tolist())

converted = pd.to_numeric(messy_numbers, errors='coerce')
print("Sau to_numeric(errors='coerce'):")
converted


### 3.4 `pd.to_datetime()` — Chuyển sang kiểu ngày giờ

Cột `Start Date` trong `df` hiện đang là dạng chuỗi (`object`) — đây là ví dụ thực tế cần chuyển đổi.


In [ ]:
df_type['Start Date'] = pd.to_datetime(df_type['Start Date'], format='%m/%d/%Y')
df_type['Last Login Time'] = pd.to_datetime(df_type['Last Login Time'], format='%I:%M %p').dt.time

print(df_type[['Start Date', 'Last Login Time']].dtypes)
df_type[['Start Date', 'Last Login Time']].head()


### 3.5 `convert_dtypes()` — Tự động suy luận kiểu dữ liệu tối ưu

Pandas sẽ tự động chọn kiểu dữ liệu "tốt nhất" (hỗ trợ `pd.NA` thay vì `NaN`/`None` hỗn hợp).


In [ ]:
df_converted = df.convert_dtypes()
df_converted.dtypes


### ✏️ 3.6 Bài tập thực hành — Change Datatype

1. Chuyển cột `Start Date` của `df` sang kiểu `datetime64[ns]`.
2. Sau khi chuyển, tạo thêm cột mới `Years Worked` = số năm đã làm việc tính đến hôm nay (gợi ý: dùng `pd.Timestamp.now()` trừ đi `Start Date`, rồi chia cho 365).
3. Chuyển cột `Team` sang kiểu `category`, in ra `df['Team'].cat.categories` để xem danh sách phòng ban.
4. Dùng `pd.to_numeric(errors='coerce')` thử áp dụng lên cột `First Name` — quan sát kết quả và giải thích tại sao gần như toàn bộ trở thành `NaN`.


In [ ]:
# TODO: Bài tập 3.6 — viết code của bạn tại đây
df_ex3 = df.copy()

# 1. Chuyển Start Date sang datetime


# 2. Tạo cột Years Worked


# 3. Chuyển Team sang category


# 4. to_numeric trên First Name


<a id="4"></a>
## 4. Xóa cột rỗng (Drop Empty Columns in Pandas)

Trong thực tế, nhiều bộ dữ liệu có các cột **hoàn toàn rỗng** (100% NaN) hoặc gần như rỗng (ví dụ >90% là NaN) — những cột này thường không mang giá trị phân tích và nên được loại bỏ.

### 4.1 Phát hiện cột rỗng hoàn toàn


In [ ]:
# Tạo ví dụ minh họa: thêm 1 cột hoàn toàn rỗng vào bản sao của df
df_empty_demo = df.copy()
df_empty_demo['Empty Col'] = np.nan

# Danh sách cột rỗng hoàn toàn
empty_cols = df_empty_demo.columns[df_empty_demo.isna().all()]
print("Các cột rỗng hoàn toàn:", list(empty_cols))


### 4.2 Xóa cột rỗng bằng `dropna(axis=1, how='all')`

In [ ]:
df_empty_demo_clean = df_empty_demo.dropna(axis=1, how='all')
print("Số cột trước:", df_empty_demo.shape[1], "-> sau:", df_empty_demo_clean.shape[1])


### 4.3 Xóa cột có tỷ lệ thiếu vượt ngưỡng cho phép

Không chỉ cột rỗng 100%, ta còn có thể loại bỏ các cột có **tỷ lệ NaN quá cao** (ví dụ trên 50%) vì chúng thường không đủ thông tin hữu ích.


In [ ]:
threshold = 0.5  # 50%
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio > threshold].index.tolist()
print("Các cột vượt ngưỡng 50% thiếu dữ liệu:", cols_to_drop)

df_dropped_cols = df.drop(columns=cols_to_drop)
print("Số cột còn lại:", df_dropped_cols.shape[1])


### ✏️ 4.4 Bài tập thực hành — Drop Empty Columns

1. Tạo bản sao `df_test` của `df`, thêm 2 cột giả hoàn toàn rỗng tên là `Note1`, `Note2`.
2. Viết code tổng quát để tự động phát hiện và xóa **mọi cột rỗng hoàn toàn** trong `df_test`.
3. Với `df` gốc (không có cột rỗng hoàn toàn), hãy in ra cột nào có tỷ lệ thiếu **trên 10%** — có cột nào không? Nếu không, giải thích tại sao ngưỡng 50% ở ví dụ trên không loại được cột nào của `df`.


In [ ]:
# TODO: Bài tập 4.4 — viết code của bạn tại đây

# 1. Tạo df_test với 2 cột rỗng


# 2. Tự động xóa cột rỗng hoàn toàn


# 3. Cột có tỷ lệ thiếu > 10%


<a id="5"></a>
## 5. Thao tác chuỗi trong Pandas (String Manipulations)

Pandas cung cấp bộ accessor `.str` để thao tác **vector hóa** trên toàn bộ cột kiểu chuỗi (nhanh hơn nhiều so với dùng vòng lặp Python thông thường).

Cú pháp chung: `df['col'].str.<phương_thức>()`


In [ ]:
# Chuẩn hóa chữ hoa/chữ thường
s = df['First Name'].dropna()

print("Viết hoa toàn bộ:   ", s.str.upper().head(3).tolist())
print("Viết thường toàn bộ:", s.str.lower().head(3).tolist())
print("Viết hoa chữ đầu:   ", s.str.title().head(3).tolist())


In [ ]:
# Loại bỏ khoảng trắng thừa ở đầu/cuối chuỗi — rất hay gặp trong dữ liệu thực tế
messy_text = pd.Series(['  Hà Nội ', 'Đà Nẵng  ', ' Huế'])
print("Gốc:", messy_text.tolist())
print("Sau strip():", messy_text.str.strip().tolist())


In [ ]:
# Tách chuỗi (split) — ví dụ tách họ và tên nếu có khoảng trắng
full_names = pd.Series(['Nguyen Van A', 'Tran Thi B', 'Le Van C'])
splitted = full_names.str.split(' ', n=1, expand=True)
splitted.columns = ['Ho', 'Ten']
splitted


In [ ]:
# Nối chuỗi (concatenate) với str.cat()
first = df['First Name'].fillna('')
team = df['Team'].fillna('Unknown')

combined = first.str.cat(team, sep=' - ')
combined.head()


In [ ]:
# Kiểm tra chuỗi con — contains(), startswith(), endswith()
print(df['Team'].dropna().str.contains('Finance').sum(), "dòng có Team chứa 'Finance'")
print(df['First Name'].dropna().str.startswith('A').sum(), "tên bắt đầu bằng chữ 'A'")


In [ ]:
# Thay thế chuỗi con — replace()
df['Team'].dropna().str.replace('Client Services', 'CS', regex=False).unique()[:5]


In [ ]:
# Trích xuất bằng biểu thức chính quy (regex) — extract()
sample_emails = pd.Series(['an.nguyen@company.com', 'binh.tran@company.com'])
usernames = sample_emails.str.extract(r'^([\w\.]+)@')
usernames


### ✏️ 5.1 Bài tập thực hành — String Manipulations

1. Chuẩn hóa cột `Team` của `df`: loại bỏ khoảng trắng thừa và chuyển về dạng "Title Case".
2. Tạo cột mới `Team Code` là 3 ký tự viết hoa đầu tiên của `Team` (ví dụ "Finance" → "FIN"). Gợi ý: kết hợp `str.upper()` và `str[:3]`.
3. Đếm xem có bao nhiêu nhân viên thuộc các Team có chứa từ "Service" (dùng `str.contains`, chú ý `na=False` để tránh lỗi với giá trị thiếu).
4. Tạo cột `Email` giả định theo định dạng `firstname.lastname@company.com` bằng cách viết thường `First Name` rồi nối thêm domain (vì không có cột Last Name, dùng luôn First Name).


In [ ]:
# TODO: Bài tập 5.1 — viết code của bạn tại đây
df_ex5 = df.copy()

# 1. Chuẩn hóa Team


# 2. Tạo cột Team Code


# 3. Đếm Team chứa 'Service'


# 4. Tạo cột Email giả định


<a id="6"></a>
## 6. Top các phương thức chuỗi thường dùng trong Pandas

Bảng tổng hợp nhanh 10 phương thức `.str` phổ biến nhất và công dụng của chúng:

| Phương thức | Công dụng |
|---|---|
| `str.lower()` / `str.upper()` | Chuyển chữ thường / chữ hoa |
| `str.strip()` | Xóa khoảng trắng thừa 2 đầu chuỗi |
| `str.len()` | Đếm độ dài chuỗi |
| `str.contains(pattern)` | Kiểm tra chuỗi có chứa `pattern` không |
| `str.replace(old, new)` | Thay thế chuỗi con |
| `str.split(sep)` | Tách chuỗi theo dấu phân cách |
| `str.startswith(prefix)` / `str.endswith(suffix)` | Kiểm tra tiền tố / hậu tố |
| `str.findall(pattern)` | Tìm tất cả các đoạn khớp với regex |
| `str.extract(pattern)` | Trích xuất nhóm khớp đầu tiên theo regex |
| `str.cat(sep)` | Nối các chuỗi lại với nhau |

Dưới đây là ví dụ minh họa nhanh cho từng phương thức trên cùng một cột dữ liệu:


In [ ]:
names = df['First Name'].dropna()

padded = '  ' + names.iloc[0] + '  '
print("1. lower():        ", names.str.lower().iloc[0])
print("2. strip():         ", repr(padded), "->", repr(padded.strip()))
print("3. len():           ", names.str.len().head(3).tolist())
print("4. contains('an'):  ", names.str.contains('an', case=False).sum(), "tên chứa 'an'")
print("5. replace('a','@'):", names.iloc[0].replace('a', '@'))
print("6. split():         ", "Nguyen Van A".split(' '))
print("7. startswith('A'): ", names.str.startswith('A').sum())
print("8. findall(vowels): ", pd.Series(names.iloc[0]).str.findall(r'[aeiouAEIOU]').iloc[0])
print("9. extract(regex):  ", pd.Series(names.iloc[0]).str.extract(r'^(\w)').iloc[0,0])
print("10. cat(sep='-'):   ", names.head(3).str.cat(sep='-'))


### ✏️ 6.1 Bài tập thực hành — String Methods

Dùng `df['First Name']` (loại bỏ NaN trước), hãy:

1. Tìm tên **dài nhất** và tên **ngắn nhất** (dùng `str.len()` kết hợp `idxmax()`/`idxmin()`).
2. Đếm số lượng tên có độ dài lớn hơn 6 ký tự.
3. Tạo cột mới chỉ chứa **ký tự đầu tiên** của mỗi tên (viết hoa).
4. Tìm tất cả các tên có chứa nguyên âm liên tiếp (2 nguyên âm đứng cạnh nhau) bằng regex.


In [ ]:
# TODO: Bài tập 6.1 — viết code của bạn tại đây
names_clean = df['First Name'].dropna()

# 1. Tên dài nhất, ngắn nhất


# 2. Đếm tên dài hơn 6 ký tự


# 3. Ký tự đầu tiên viết hoa


# 4. Tên chứa 2 nguyên âm liên tiếp


<a id="7"></a>
## 7. Phát hiện và xử lý kiểu dữ liệu hỗn hợp (Detect Mixed Data Types and Fix it)

**Mixed data type** xảy ra khi một cột (thường có `dtype = object`) chứa **nhiều kiểu dữ liệu Python khác nhau** cùng lúc — ví dụ vừa có `int`, vừa có `str`, vừa có `float`, `None`... Đây là nguồn lỗi rất phổ biến khi tính toán hoặc sắp xếp dữ liệu.

### 7.1 Nguyên nhân
- Nhập liệu thủ công không nhất quán (vd: `"100"`, `100`, `"100kg"`)
- Ghép nhiều nguồn dữ liệu có định dạng khác nhau
- Lỗi khi đọc file (Excel/CSV) tự động suy luận kiểu sai

### 7.2 Phát hiện kiểu dữ liệu hỗn hợp

Cách phổ biến nhất: dùng `apply(type)` để xem kiểu Python thực sự của từng giá trị trong cột.


In [ ]:
# Tạo dữ liệu minh họa có kiểu hỗn hợp
mixed_series = pd.Series([100, "200", 300.5, "abc", None, True])
print(mixed_series)
print("\nKiểu dữ liệu Python của từng phần tử:")
print(mixed_series.apply(type))


In [ ]:
# Đếm số lượng kiểu dữ liệu khác nhau xuất hiện trong cột
mixed_series.apply(type).value_counts()


### 7.3 Sửa kiểu dữ liệu hỗn hợp

Chiến lược tùy theo mục tiêu:
- Nếu muốn **ép về số**: dùng `pd.to_numeric(errors='coerce')`
- Nếu muốn **ép về chuỗi** (đồng nhất định dạng): dùng `.astype(str)`
- Dùng `infer_objects()` để Pandas tự suy luận lại kiểu phù hợp nhất sau khi đã làm sạch


In [ ]:
# Ép toàn bộ về số, giá trị không hợp lệ -> NaN
fixed_numeric = pd.to_numeric(mixed_series, errors='coerce')
print(fixed_numeric)
print("dtype:", fixed_numeric.dtype)


In [ ]:
# Ép toàn bộ về chuỗi để đồng nhất định dạng hiển thị
fixed_str = mixed_series.astype(str)
print(fixed_str)
print("dtype:", fixed_str.dtype)


In [ ]:
# infer_objects(): để Pandas tự suy luận kiểu tối ưu sau khi loại bỏ phần tử gây lỗi
clean_series = pd.Series([1, 2, 3, 4])  # sau khi đã làm sạch, không còn hỗn hợp
print(clean_series.dtype)
print(clean_series.infer_objects().dtype)


### 7.4 Áp dụng kiểm tra mixed type cho toàn bộ DataFrame


In [ ]:
def check_mixed_types(dataframe):
    '''Trả về danh sách các cột có từ 2 kiểu dữ liệu Python trở lên (bỏ qua NaN).'''
    mixed_cols = []
    for col in dataframe.columns:
        types_in_col = dataframe[col].dropna().apply(type).unique()
        if len(types_in_col) > 1:
            mixed_cols.append((col, list(types_in_col)))
    return mixed_cols

check_mixed_types(df)


> Trong `employees.csv`, kiểu dữ liệu trong mỗi cột khá đồng nhất (Pandas đã tự suy luận đúng khi đọc CSV), nên hàm trên có thể trả về danh sách rỗng — điều đó là **bình thường** và cũng là một cách để *xác nhận* dữ liệu sạch về mặt kiểu dữ liệu.

### ✏️ 7.5 Bài tập thực hành — Mixed Data Types

1. Tạo một `Series` mẫu gồm ít nhất 6 phần tử với ít nhất 3 kiểu dữ liệu Python khác nhau (int, str, float, bool, None...).
2. Viết hàm `check_mixed_types` áp dụng lên `Series` đó (từng phần tử) — in ra kiểu dữ liệu của mỗi phần tử.
3. Chuẩn hóa `Series` trên về kiểu số (`float`) bằng `pd.to_numeric(errors='coerce')`, sau đó điền các giá trị `NaN` sinh ra bằng `0`.
4. Chạy hàm `check_mixed_types(df)` ở trên với `df` gốc — nếu không có cột nào bị hỗn hợp, hãy **cố tình** thêm 1 cột mới có giá trị hỗn hợp vào bản sao của `df` rồi kiểm tra lại để xác nhận hàm hoạt động đúng.


In [ ]:
# TODO: Bài tập 7.5 — viết code của bạn tại đây

# 1. Tạo Series mẫu có kiểu hỗn hợp


# 2. Kiểm tra kiểu của từng phần tử


# 3. Chuẩn hóa về float, điền NaN bằng 0


# 4. Thêm cột hỗn hợp giả vào bản sao của df và kiểm tra lại


<a id="8"></a>
## 8. 🏆 Bài tập tổng hợp (Capstone Project)

Bây giờ hãy áp dụng **toàn bộ kiến thức** ở các phần 1–7 để xây dựng một **pipeline làm sạch dữ liệu hoàn chỉnh** cho `employees.csv`. Hãy viết một hàm `clean_employees(df)` thực hiện tuần tự các bước sau và trả về DataFrame đã sạch:

1. **Xóa cột rỗng hoàn toàn** (nếu có).
2. **Chuẩn hóa chuỗi**: loại bỏ khoảng trắng thừa ở `First Name` và `Team`.
3. **Xử lý thiếu dữ liệu**:
   - `First Name` thiếu → điền `"Unknown"`
   - `Gender` thiếu → điền `"Not Specified"`
   - `Team` thiếu → điền `"Unassigned"`
   - `Senior Management` thiếu → điền `False`
4. **Chuyển đổi kiểu dữ liệu**:
   - `Start Date` → `datetime64[ns]`
   - `Senior Management` → `bool`
   - `Team`, `Gender` → `category`
5. **Loại bỏ trùng lặp** hoàn toàn (nếu có).
6. **Kiểm tra kiểu dữ liệu hỗn hợp** còn sót lại bằng `check_mixed_types()` (đã viết ở phần 7).
7. In ra **báo cáo tóm tắt** trước/sau khi làm sạch: số dòng, số cột, tổng số giá trị thiếu, số dòng trùng lặp.

> 💡 Đây là bài tập mở — không có 1 đáp án duy nhất, quan trọng là quy trình logic và code chạy được.


In [ ]:
# TODO: Capstone Project — viết hàm clean_employees() tại đây

def clean_employees(input_df: pd.DataFrame) -> pd.DataFrame:
    data = input_df.copy()

    # Bước 1: Xóa cột rỗng hoàn toàn


    # Bước 2: Chuẩn hóa chuỗi


    # Bước 3: Xử lý thiếu dữ liệu


    # Bước 4: Chuyển đổi kiểu dữ liệu


    # Bước 5: Loại bỏ trùng lặp


    return data


# Chạy thử pipeline
df_before = df.copy()
df_clean = clean_employees(df)

print("=== BÁO CÁO TRƯỚC KHI LÀM SẠCH ===")
print("Kích thước:", df_before.shape)
print("Tổng giá trị thiếu:", df_before.isna().sum().sum())
print("Số dòng trùng lặp:", df_before.duplicated().sum())

print("\n=== BÁO CÁO SAU KHI LÀM SẠCH ===")
print("Kích thước:", df_clean.shape)
print("Tổng giá trị thiếu:", df_clean.isna().sum().sum())
print("Số dòng trùng lặp:", df_clean.duplicated().sum())


<a id="9"></a>
## 9. Tổng kết & Checklist làm sạch dữ liệu

Khi làm sạch một bộ dữ liệu mới, bạn có thể dùng checklist sau như một quy trình chuẩn:

- [ ] **Khám phá dữ liệu**: `df.info()`, `df.describe()`, `df.head()`
- [ ] **Kiểm tra giá trị thiếu**: `df.isna().sum()` → quyết định xóa (`dropna`) hay điền (`fillna`/`interpolate`)
- [ ] **Kiểm tra & xóa cột rỗng / gần rỗng**: `dropna(axis=1, how='all')` hoặc theo ngưỡng tỷ lệ thiếu
- [ ] **Kiểm tra dữ liệu trùng lặp**: `duplicated()` → `drop_duplicates()`
- [ ] **Chuẩn hóa kiểu dữ liệu**: `astype()`, `to_numeric()`, `to_datetime()`, `convert_dtypes()`
- [ ] **Chuẩn hóa chuỗi**: `str.strip()`, `str.lower()/upper()`, loại bỏ ký tự đặc biệt, chuẩn hóa định dạng
- [ ] **Phát hiện kiểu dữ liệu hỗn hợp** trong các cột `object`: `apply(type)`, sau đó ép kiểu đồng nhất
- [ ] **Kiểm tra lại lần cuối** trước khi phân tích / mô hình hóa: `df.info()`, `df.isna().sum()`, `df.duplicated().sum()`

### 📌 Tài liệu tham khảo gốc
1. [Handling Missing Data](https://www.geeksforgeeks.org/data-analysis/working-with-missing-data-in-pandas/)
2. [Removing Duplicates](https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-drop_duplicates/)
3. [Pandas Change Datatype](https://www.geeksforgeeks.org/pandas/pandas-change-datatype/)
4. [Drop Empty Columns in Pandas](https://www.geeksforgeeks.org/python/drop-empty-columns-in-pandas/)
5. [String manipulations in Pandas](https://www.geeksforgeeks.org/python/string-manipulations-in-pandas-dataframe/)
6. [String methods in Pandas](https://www.geeksforgeeks.org/python/top-10-string-methods-in-pandas/)
7. [Detect Mixed Data Types and Fix it](https://www.geeksforgeeks.org/python/pandas-detect-mixed-data-types-and-fix-it/)

---
**Chúc bạn học tốt! 🎉** Hãy thử hoàn thành tất cả các bài tập trước khi xem đáp án hoặc tìm kiếm thêm thông tin.
